# 07 -- Sensitivity and Robustness: the n/h Label-Threshold Sweep

**Question:** NB06 found the "High" tier (predicted 2+ future stagnant skills, top-10% within-`t`
rank) is not distinguishable from random triage overall, and *inverts* for DD learners (lift 0.75)
-- see `dev_docs/notebook_findings/06_target_b_findings.md` Sec5. That failure was diagnosed against
one label definition: `h=5, n=4`. `n` and `h` are both `[SENS]` sweep parameters (DATASET_CARD.md
Sec6) with an explicitly open owner decision (3-Q1). This notebook asks: **does a different point on
that sweep produce a label with enough class separation to support a shippable High tier**, and
separately, is the modelling pipeline itself robust to the horizon `h`?

Kernel `Python (c3-ews)`, seed `20260828`. Reuses NB05's exact LogisticRegression configuration
(`class_weight='balanced'`, median-impute + standard-scale preprocessing) and NB05/NB06's grouped
210/90 `student_id` train/test split -- **not re-tuned, not re-split, not re-featured.** Only the
label (`future_stagnant_count`) changes across arms; features are identical throughout.

**Arms (kept unconfounded -- n and h are never compared against each other simultaneously):**

- **n-sweep, all at h=5:** `labels_sweep/*_sweepn{2,3,4,6,8}.csv` (n=4 is the primary arm, reproduced
  here for a clean baseline row -- not re-tuned).
- **h-sweep, all at n=8:** `sweepn8` (the h=5 point) plus `labels_robustness/*_h{3,7}.csv`. The
  primary table (h=5, n=4) is **never** compared directly against the h=3/h=7 robustness files --
  that would confound `h` and `n` (n=4 vs n=8). The h=5 point of the h-sweep is `sweepn8`, not the
  primary table.

**Out of scope:** no re-tuning, no feature changes, no revisiting model selection, no changing the
primary arm. This notebook only asks whether a different label recipe changes the answer -- it does
not re-answer NB05's modelling question.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd

from nbconfig import *

np.random.seed(RANDOM_SEED)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
print("seed:", RANDOM_SEED)
print("dataset dir:", DATASET_DIR)
print("tables dir:", TABLES_DIR)

seed: 20260828
dataset dir: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\dataset
tables dir: C:\Users\Yasindu\Desktop\ResearchProject\Componant 3 -Teachers Dashboard\Learning-Insight-Early-Warning-System\notebooks\outputs\tables


## A. Setup -- load the shared feature matrix and the primary training table

Features are identical across every arm in this notebook; only the label changes. We reuse NB05's
`24_engineered_feature_matrix.csv` (2,700 rows, the feature space + the grouped `split` column) as
the feature source, and join each arm's own label file's `future_stagnant_count` onto it by
`(student_id, t)`. `split` (and therefore the 210/90 grouped train/test partition) comes from this
one table and is reused unmodified for every arm -- we never re-split per arm.

In [2]:
FEATURE_MATRIX_TABLE = TABLES_DIR / "24_engineered_feature_matrix.csv"

feat = pd.read_csv(FEATURE_MATRIX_TABLE)
META_COLS = ["student_id", "t", "split", "future_stagnant_count"]
FEATURES_A = [c for c in feat.columns if c not in META_COLS]
print(f"Feature matrix: {feat.shape[0]} rows, {len(FEATURES_A)} features.")

SPLIT_MAP = feat[["student_id", "t", "split"]].copy()
train_ids = set(feat.loc[feat.split == "train", "student_id"])
test_ids = set(feat.loc[feat.split == "test", "student_id"])
assert len(train_ids) == 210 and len(test_ids) == 90
assert train_ids & test_ids == set(), "student_id leaked across the split"
print(f"Reused grouped split: {len(train_ids)} train learners / {len(test_ids)} test learners. No overlap.")

audit = pd.read_csv(AUDIT_LABELS_PATH)[["student_id", "population"]]
assert audit.student_id.nunique() == 300
print("Audit population labels loaded for evaluation-only stratification (never a feature).")

Feature matrix: 2700 rows, 58 features.
Reused grouped split: 210 train learners / 90 test learners. No overlap.
Audit population labels loaded for evaluation-only stratification (never a feature).


In [3]:
# Sanity check: the primary training table equals labels_sweep n=4 (sweepn4) on all 2700 rows.
LABELS_SWEEP_DIR = DATASET_DIR / "labels_sweep"
LABELS_ROBUST_DIR = DATASET_DIR / "labels_robustness"

training_table = pd.read_csv(TRAINING_TABLE_PATH)
sweepn4 = pd.read_csv(LABELS_SWEEP_DIR / "labels_at_t_seed20260828_n300_h5_sweepn4.csv")

check = training_table.merge(
    sweepn4[["student_id", "t", "future_stagnant_count"]],
    on=["student_id", "t"], suffixes=("", "_sweepn4"),
)
assert len(check) == 2700 == len(training_table) == len(sweepn4)
assert (check["future_stagnant_count"] == check["future_stagnant_count_sweepn4"]).all()
print("PASS: primary training table's future_stagnant_count matches labels_sweep n=4 (sweepn4) "
      f"exactly on all {len(check)} rows.")

PASS: primary training table's future_stagnant_count matches labels_sweep n=4 (sweepn4) exactly on all 2700 rows.


## B. Arm loader -- join, collapse, verify split integrity

For every arm: join the arm's label file onto the shared feature matrix by `(student_id, t)`
(inner join -- h=3/h=7 have different valid-`t` ranges than the h=5 arms, so row counts legitimately
differ across arms), collapse `future_stagnant_count` to `{0, 1, 2+}` exactly as NB05 did (`clip(upper=2)`),
and re-assert zero learner overlap between train/test for that arm's surviving rows.

In [4]:
CLASS_NAMES = {0: "0", 1: "1", 2: "2+"}
CLASSES = [0, 1, 2]

def load_arm(label_path, arm_name):
    lbl = pd.read_csv(label_path)[["student_id", "t", "future_stagnant_count"]]
    df = feat[["student_id", "t", "split"] + FEATURES_A].merge(lbl, on=["student_id", "t"], how="inner")
    df["target"] = df["future_stagnant_count"].clip(upper=2).astype(int)
    df = df.merge(audit, on="student_id", how="left")
    assert df["population"].isna().sum() == 0

    train = df[df.split == "train"].reset_index(drop=True)
    test = df[df.split == "test"].reset_index(drop=True)
    assert set(train.student_id) & set(test.student_id) == set(), f"{arm_name}: student_id leaked across split"

    return {
        "name": arm_name, "df": df, "train": train, "test": test,
        "n_rows": len(df), "n_train_learners": train.student_id.nunique(),
        "n_test_learners": test.student_id.nunique(),
    }

N_SWEEP_FILES = {
    2: LABELS_SWEEP_DIR / "labels_at_t_seed20260828_n300_h5_sweepn2.csv",
    3: LABELS_SWEEP_DIR / "labels_at_t_seed20260828_n300_h5_sweepn3.csv",
    4: LABELS_SWEEP_DIR / "labels_at_t_seed20260828_n300_h5_sweepn4.csv",
    6: LABELS_SWEEP_DIR / "labels_at_t_seed20260828_n300_h5_sweepn6.csv",
    8: LABELS_SWEEP_DIR / "labels_at_t_seed20260828_n300_h5_sweepn8.csv",
}
H_SWEEP_FILES = {
    3: LABELS_ROBUST_DIR / "labels_at_t_seed20260828_n300_h3.csv",
    5: LABELS_SWEEP_DIR / "labels_at_t_seed20260828_n300_h5_sweepn8.csv",  # h=5 point of the h-sweep IS sweepn8, at matched n=8
    7: LABELS_ROBUST_DIR / "labels_at_t_seed20260828_n300_h7.csv",
}

n_arms = {n: load_arm(p, f"n={n} (h=5)") for n, p in N_SWEEP_FILES.items()}
h_arms = {h: load_arm(p, f"h={h} (n=8)") for h, p in H_SWEEP_FILES.items()}

print("n-sweep arms (all h=5):")
for n, a in n_arms.items():
    print(f"  n={n}: {a['n_rows']} rows, {a['n_train_learners']}/{a['n_test_learners']} train/test learners, "
          f"target dist {a['df'].target.value_counts().sort_index().to_dict()}")

print("\nh-sweep arms (all n=8 -- CRITICAL: never compared against the primary n=4 table):")
for h, a in h_arms.items():
    print(f"  h={h}: {a['n_rows']} rows, {a['n_train_learners']}/{a['n_test_learners']} train/test learners, "
          f"target dist {a['df'].target.value_counts().sort_index().to_dict()}")

n-sweep arms (all h=5):
  n=2: 2700 rows, 210/90 train/test learners, target dist {0: 495, 1: 1275, 2: 930}
  n=3: 2700 rows, 210/90 train/test learners, target dist {0: 954, 1: 1371, 2: 375}
  n=4: 2700 rows, 210/90 train/test learners, target dist {0: 1213, 1: 1326, 2: 161}
  n=6: 2700 rows, 210/90 train/test learners, target dist {0: 1554, 1: 1110, 2: 36}
  n=8: 2700 rows, 210/90 train/test learners, target dist {0: 1812, 1: 881, 2: 7}

h-sweep arms (all n=8 -- CRITICAL: never compared against the primary n=4 table):
  h=3: 2700 rows, 210/90 train/test learners, target dist {0: 2102, 1: 598}
  h=5: 2700 rows, 210/90 train/test learners, target dist {0: 1812, 1: 881, 2: 7}
  h=7: 2400 rows, 210/90 train/test learners, target dist {0: 1395, 1: 983, 2: 22}


In [5]:
# Sanity assertion: at matched n=8, the label must be monotone non-decreasing in h, with ZERO
# violations. Comparing h=3/7 (n=8) against the PRIMARY table (n=4) instead would be the n/h
# confound the spec warns against and produces ~466 spurious violations -- we do not make that mistake.
h3_lbl = pd.read_csv(H_SWEEP_FILES[3])[["student_id", "t", "future_stagnant_count"]].rename(columns={"future_stagnant_count": "h3"})
h5_lbl = pd.read_csv(H_SWEEP_FILES[5])[["student_id", "t", "future_stagnant_count"]].rename(columns={"future_stagnant_count": "h5"})
h7_lbl = pd.read_csv(H_SWEEP_FILES[7])[["student_id", "t", "future_stagnant_count"]].rename(columns={"future_stagnant_count": "h7"})

mono = h3_lbl.merge(h5_lbl, on=["student_id", "t"]).merge(h7_lbl, on=["student_id", "t"])
viol_35 = int((mono.h5 < mono.h3).sum())
viol_57 = int((mono.h7 < mono.h5).sum())
print(f"Matched-n=8 monotonicity check over {len(mono)} rows: h5<h3 violations={viol_35}, h7<h5 violations={viol_57}")
assert viol_35 == 0 and viol_57 == 0, "future_stagnant_count is not monotone non-decreasing in h at matched n=8"
print("PASS: future_stagnant_count is monotone non-decreasing in h at matched n=8, zero violations.")

# Contrast: what the WRONG (confounded) comparison would look like, reported for completeness only.
primary_lbl = pd.read_csv(TRAINING_TABLE_PATH)[["student_id", "t", "future_stagnant_count"]].rename(columns={"future_stagnant_count": "primary_n4h5"})
wrong = h3_lbl.merge(primary_lbl, on=["student_id", "t"]).merge(h7_lbl, on=["student_id", "t"])
wrong_viol = int((wrong.primary_n4h5 < wrong.h3).sum() + (wrong.h7 < wrong.primary_n4h5).sum())
print(f"\n[DO NOT USE] confounded n=4-vs-n=8 comparison would show {wrong_viol} spurious 'violations' "
      "-- this is the n/h confound the spec warns against, shown here only to document why it is wrong.")

Matched-n=8 monotonicity check over 2400 rows: h5<h3 violations=0, h7<h5 violations=0
PASS: future_stagnant_count is monotone non-decreasing in h at matched n=8, zero violations.

[DO NOT USE] confounded n=4-vs-n=8 comparison would show 466 spurious 'violations' -- this is the n/h confound the spec warns against, shown here only to document why it is wrong.


## C. Shared pipeline (reused verbatim from NB05 -- not re-tuned here)

Identical `LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_SEED)`
inside a median-impute + standard-scale `Pipeline`, exactly as NB05 Sec C/D.

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score, precision_score, recall_score

def pre_steps():
    return [("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]

def logreg_pipe():
    return Pipeline(pre_steps() + [
        ("clf", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_SEED))
    ])

def fit_and_score(arm):
    train, test = arm["train"], arm["test"]
    pipe = logreg_pipe()
    pipe.fit(train[FEATURES_A], train.target)
    pred = pipe.predict(test[FEATURES_A])
    proba = pipe.predict_proba(test[FEATURES_A])
    arm["pipe"] = pipe
    arm["pred"] = pred
    arm["proba"] = proba
    return arm

def baseline_scores(arm):
    train, test = arm["train"], arm["test"]
    # majority-class baseline
    maj = DummyClassifier(strategy="most_frequent", random_state=RANDOM_SEED)
    maj.fit(train[FEATURES_A], train.target)
    maj_pred = maj.predict(test[FEATURES_A])
    # t-only baseline: single-feature logistic regression on t
    t_pipe = logreg_pipe()
    t_pipe.fit(train[["t"]], train.target)
    t_pred = t_pipe.predict(test[["t"]])
    return {
        "majority_macro_f1": f1_score(test.target, maj_pred, average="macro"),
        "majority_bal_acc": balanced_accuracy_score(test.target, maj_pred),
        "t_only_macro_f1": f1_score(test.target, t_pred, average="macro"),
        "t_only_bal_acc": balanced_accuracy_score(test.target, t_pred),
    }

for arm in list(n_arms.values()) + list(h_arms.values()):
    fit_and_score(arm)
print("Fitted the primary pipeline on every arm (n-sweep and h-sweep).")

Fitted the primary pipeline on every arm (n-sweep and h-sweep).


## D. High tier -- reuse NB06's within-`t` top-10% rank

Reused mechanism: rank each learner's predicted P(2+) **within their own snapshot `t`** among the
other test learners at that `t`, then flag the top 10% as High. Simplification versus NB06: NB06
additionally suppressed High before `t=9` because 2+ prevalence was ~0 that early *for the primary
n=4/h=5 arm specifically*; that gate was calibrated to one arm's prevalence curve and is not
re-derived here for every sweep arm (out of scope -- no re-tuning). We therefore report the top-10%
within-`t` rank uniformly across all `t` for every arm; this is a deliberate scope simplification,
noted here rather than silently applied. Arms where a class is entirely absent (h=3 has no 2+ rows
at all) skip the High-tier evaluation and are reported on the ≥1 task only, per the spec.

In [7]:
def build_high_tier(arm, high_frac=0.10):
    test = arm["test"].copy()
    proba = arm["proba"]
    classes = list(arm["pipe"].named_steps["clf"].classes_)
    if 2 not in classes:
        # class 2 (2+) never occurs in this arm's training data -- High tier is structurally
        # undefined, not just underpowered (this is the h=3 case).
        test["score_2plus"] = np.nan
        test["risk_pct"] = np.nan
        test["high_flag"] = False
        arm["scored_test"] = test
        arm["high_tier_defined"] = False
        return test
    class2_idx = classes.index(2)
    test["score_2plus"] = proba[:, class2_idx]
    test["risk_pct"] = test.groupby("t")["score_2plus"].rank(pct=True, method="average")
    high_cut = 1 - high_frac
    test["high_flag"] = test["risk_pct"] >= high_cut
    arm["scored_test"] = test
    arm["high_tier_defined"] = True
    return test

def high_tier_metrics(arm):
    test = arm["scored_test"]
    y2 = test.target == 2
    if not arm.get("high_tier_defined", False) or y2.sum() == 0:
        return None  # no 2+ rows at all -- High tier undefined (h=3)
    prevalence = y2.mean()
    high = test[test.high_flag]
    precision = y2[test.high_flag].mean() if test.high_flag.sum() else np.nan
    recall = (y2 & test.high_flag).sum() / y2.sum() if y2.sum() else np.nan
    lift = precision / prevalence if prevalence else np.nan
    return {"n_high": int(test.high_flag.sum()), "precision": precision, "recall": recall,
            "prevalence": prevalence, "lift": lift}

for arm in list(n_arms.values()) + list(h_arms.values()):
    build_high_tier(arm)
    arm["high_metrics"] = high_tier_metrics(arm)
print("High tier (top-10% within t) built for every arm.")

High tier (top-10% within t) built for every arm.


## E. Learner-clustered bootstrap CI on (High-tier precision − prevalence)

4,000 resamples over **learners** (`student_id`), not rows -- the credible-evidence test used in
`06_target_b_findings.md` Sec1. Resample test-set learners with replacement, pool their flagged rows,
recompute precision and prevalence on the pooled resample, and report the 95% CI of the difference.

In [8]:
def bootstrap_precision_minus_prevalence(arm, n_boot=4000, seed=RANDOM_SEED):
    test = arm["scored_test"]
    if arm["high_metrics"] is None:
        return None
    learners = test.student_id.unique()
    rng = np.random.RandomState(seed)
    diffs = np.empty(n_boot)
    for b in range(n_boot):
        sampled = rng.choice(learners, size=len(learners), replace=True)
        rows = test[test.student_id.isin(sampled)]
        # weight by how many times each learner was drawn
        counts = pd.Series(sampled).value_counts()
        rows = rows.merge(counts.rename("w"), left_on="student_id", right_index=True)
        rows = rows.loc[rows.index.repeat(rows.w)]
        y2 = rows.target == 2
        prevalence = y2.mean()
        flagged = rows[rows.high_flag]
        precision = (flagged.target == 2).mean() if len(flagged) else np.nan
        diffs[b] = precision - prevalence if prevalence == prevalence else np.nan
    diffs = diffs[~np.isnan(diffs)]
    return {
        "point_estimate": float(np.mean(diffs)),
        "ci_low": float(np.percentile(diffs, 2.5)),
        "ci_high": float(np.percentile(diffs, 97.5)),
        "p_better_than_base_rate": float((diffs > 0).mean()),
        "n_boot_valid": len(diffs),
    }

for arm in list(n_arms.values()) + list(h_arms.values()):
    arm["bootstrap_ci"] = bootstrap_precision_minus_prevalence(arm)
print("Learner-clustered bootstrap complete (4000 resamples) for every arm with a defined 2+ class.")

Learner-clustered bootstrap complete (4000 resamples) for every arm with a defined 2+ class.


## F. Per-population (DD/MLD/TD) High-tier lift, for every arm

Reused from NB06 Sec E.1 -- population is from the quarantined audit labels, evaluation only.

In [9]:
def per_population_lift(arm):
    test = arm["scored_test"]
    if arm["high_metrics"] is None:
        return pd.DataFrame(columns=["population", "n", "n_flagged", "precision", "recall", "prevalence", "lift"])
    rows = []
    for pop, grp in test.groupby("population"):
        y2 = grp.target == 2
        prevalence = y2.mean()
        flagged = grp[grp.high_flag]
        precision = (flagged.target == 2).mean() if len(flagged) else np.nan
        recall = (y2 & grp.high_flag).sum() / y2.sum() if y2.sum() else np.nan
        rows.append({"population": pop, "n": len(grp), "n_flagged": int(grp.high_flag.sum()),
                     "precision": precision, "recall": recall, "prevalence": prevalence,
                     "lift": precision / prevalence if prevalence else np.nan})
    return pd.DataFrame(rows)

for arm in list(n_arms.values()) + list(h_arms.values()):
    arm["per_pop"] = per_population_lift(arm)

print("Per-population lift (n-sweep):")
for n, arm in n_arms.items():
    print(f"\n-- n={n} --")
    print(arm["per_pop"].to_string(index=False) if len(arm["per_pop"]) else "  (High tier undefined -- no 2+ rows)")

Per-population lift (n-sweep):

-- n=2 --
population   n  n_flagged  precision   recall  prevalence     lift
        DD  90         21   0.428571 0.230769    0.433333 0.989011
       MLD 117         21   0.571429 0.413793    0.247863 2.305419
        TD 603         48   0.437500 0.098592    0.353234 1.238556

-- n=3 --
population   n  n_flagged  precision   recall  prevalence     lift
        DD  90         16   0.125000 0.142857    0.155556 0.803571
       MLD 117         18   0.222222 0.400000    0.085470 2.600000
        TD 603         56   0.178571 0.112360    0.147595 1.209872

-- n=4 --
population   n  n_flagged  precision   recall  prevalence     lift
        DD  90         20   0.100000 0.200000    0.111111 0.900000
       MLD 117         14   0.285714 0.666667    0.051282 5.571429
        TD 603         56   0.053571 0.085714    0.058043 0.922959

-- n=6 --
population   n  n_flagged  precision  recall  prevalence     lift
        DD  90          6   0.000000     0.0    0.02222

## G. "≥1 watch" signal -- lift at t=3, every arm

This is the signal NB06's findings concluded WAS supported (lift 2.70 at `t=3` for the primary
arm). Reported here as the robustness check for that finding across every sweep arm.

In [10]:
def watch_lift_at_t3(arm):
    test = arm["test"].copy()
    proba = arm["proba"]
    # P(>=1) = 1 - P(class 0)
    test["p_ge1"] = 1 - proba[:, 0]
    t3 = test[test.t == 3]
    if len(t3) == 0:
        return np.nan, np.nan, np.nan
    prevalence = (t3.target >= 1).mean()
    # flag the same count as would be flagged by predicted class >= 1
    flagged = t3["p_ge1"] >= 0.5
    if flagged.sum() == 0:
        return np.nan, prevalence, np.nan
    precision = (t3.loc[flagged, "target"] >= 1).mean()
    lift = precision / prevalence if prevalence else np.nan
    return precision, prevalence, lift

watch_rows = []
for n, arm in n_arms.items():
    prec, prev, lift = watch_lift_at_t3(arm)
    watch_rows.append({"arm": f"n={n} (h=5)", "t3_precision": prec, "t3_prevalence": prev, "t3_lift": lift})
for h, arm in h_arms.items():
    prec, prev, lift = watch_lift_at_t3(arm)
    watch_rows.append({"arm": f"h={h} (n=8)", "t3_precision": prec, "t3_prevalence": prev, "t3_lift": lift})

watch_table = pd.DataFrame(watch_rows)
watch_table.to_csv(TABLES_DIR / "80_watch_signal_t3_lift_by_arm.csv", index=False)
watch_table

,arm,t3_precision,t3_prevalence,t3_lift
0,n=2 (h=5),0.617647,0.600000,1.029412
1,n=3 (h=5),0.500000,0.333333,1.500000
2,n=4 (h=5),0.750000,0.277778,2.700000
3,n=6 (h=5),0.250000,0.222222,1.125000
4,n=8 (h=5),0.272727,0.200000,1.363636
5,h=3 (n=8),0.222222,0.111111,2.000000
6,h=5 (n=8),0.272727,0.200000,1.363636
7,h=7 (n=8),0.500000,0.388889,1.285714


## H. Headline table -- n-sweep macro-F1, High-tier lift and CI

n=3 has a 2+ base rate of ~13.9% versus the primary arm's 6.0% -- inside DATASET_CARD.md Sec6's
expected 8-15% "High" band (measured there for the ≥1 rate; the 2+ rate is the quantity that
determines whether the High tier itself has enough positives to be learnable). n=2 is
**sensitivity-only and excluded from any product recommendation** -- DATASET_CARD.md Sec6 flags it
as near-degenerate (≥1 positive rate ~81.7%).

In [11]:
summary_rows = []
for n, arm in n_arms.items():
    base = baseline_scores(arm)
    macro_f1 = f1_score(arm["test"].target, arm["pred"], average="macro")
    bal_acc = balanced_accuracy_score(arm["test"].target, arm["pred"])
    hm = arm["high_metrics"]
    ci = arm["bootstrap_ci"]
    two_plus_rate = (arm["df"].target == 2).mean()
    ge1_rate = (arm["df"].target >= 1).mean()
    summary_rows.append({
        "arm": f"n={n}", "n_rows": arm["n_rows"], "2plus_rate": two_plus_rate, "ge1_rate": ge1_rate,
        "macro_F1": macro_f1, "macro_F1_majority_baseline": base["majority_macro_f1"],
        "macro_F1_t_only_baseline": base["t_only_macro_f1"], "balanced_acc": bal_acc,
        "balanced_acc_majority_baseline": base["majority_bal_acc"],
        "balanced_acc_t_only_baseline": base["t_only_bal_acc"],
        "high_precision": hm["precision"] if hm else np.nan,
        "high_recall": hm["recall"] if hm else np.nan,
        "high_lift": hm["lift"] if hm else np.nan,
        "ci_precision_minus_prevalence": ci["point_estimate"] if ci else np.nan,
        "ci_low": ci["ci_low"] if ci else np.nan,
        "ci_high": ci["ci_high"] if ci else np.nan,
        "p_better_than_base_rate": ci["p_better_than_base_rate"] if ci else np.nan,
        "sensitivity_only": n == 2,
    })

n_sweep_summary = pd.DataFrame(summary_rows)
n_sweep_summary.to_csv(TABLES_DIR / "81_n_sweep_headline_summary.csv", index=False)
n_sweep_summary

,arm,n_rows,2plus_rate,ge1_rate,macro_F1,macro_F1_majority_baseline,macro_F1_t_only_baseline,balanced_acc,balanced_acc_majority_baseline,balanced_acc_t_only_baseline,high_precision,high_recall,high_lift,ci_precision_minus_prevalence,ci_low,ci_high,p_better_than_base_rate,sensitivity_only
0,n=2,2700,0.344444,0.816667,0.446584,0.217041,0.381160,0.475873,0.333333,0.484517,0.466667,0.149466,1.345196,0.120163,0.008956,0.237678,0.98425,True
1,n=3,2700,0.138889,0.646667,0.426749,0.231884,0.346501,0.477566,0.333333,0.471418,0.177778,0.141593,1.274336,0.037063,-0.030601,0.108642,0.85375,False
2,n=4,2700,0.059630,0.550741,0.423929,0.229773,0.323021,0.477209,0.333333,0.422724,0.100000,0.176471,1.588235,0.036490,-0.022780,0.106691,0.86650,False
3,n=6,2700,0.013333,0.424444,0.375520,0.230479,0.256037,0.377558,0.333333,0.356457,0.011111,0.076923,0.692308,-0.004808,-0.023457,0.021088,0.30475,False
4,n=8,2700,0.002593,0.328889,0.408886,0.258812,0.196867,0.406138,0.333333,0.516213,0.000000,0.000000,0.000000,-0.001262,-0.003704,0.000000,0.00000,False


**n=2 is reported above for completeness only** -- it is marked `sensitivity_only=True` and
must not be used for a product recommendation (near-degenerate ≥1 positive rate ~81.7%,
DATASET_CARD.md Sec6 warning).

## I. h-sweep (robustness only, not a product arm)

All three points sit at n=8 -- the weakest-signal n in the sweep (2+ prevalence ~0.000-0.009 across
this arm's snapshots). This is a **pipeline robustness check** (does the modelling pipeline still
behave sensibly as the horizon changes), not a candidate arm for the product -- these near-zero
metrics are not presented as viable. `h=3` has **no 2+ rows at all** in this dataset, so only the
≥1 task is reported for it; a High tier cannot be defined when a class does not exist.

In [12]:
h_summary_rows = []
for h, arm in h_arms.items():
    base = baseline_scores(arm)
    macro_f1 = f1_score(arm["test"].target, arm["pred"], average="macro")
    bal_acc = balanced_accuracy_score(arm["test"].target, arm["pred"])
    hm = arm["high_metrics"]
    ci = arm["bootstrap_ci"]
    two_plus_rate = (arm["df"].target == 2).mean()
    ge1_rate = (arm["df"].target >= 1).mean()
    h_summary_rows.append({
        "arm": f"h={h} (n=8)", "n_rows": arm["n_rows"], "2plus_rate": two_plus_rate, "ge1_rate": ge1_rate,
        "macro_F1": macro_f1, "macro_F1_majority_baseline": base["majority_macro_f1"],
        "balanced_acc": bal_acc, "balanced_acc_majority_baseline": base["majority_bal_acc"],
        "high_precision": hm["precision"] if hm else np.nan,
        "high_recall": hm["recall"] if hm else np.nan,
        "high_lift": hm["lift"] if hm else np.nan,
        "ci_precision_minus_prevalence": ci["point_estimate"] if ci else np.nan,
        "ci_low": ci["ci_low"] if ci else np.nan,
        "ci_high": ci["ci_high"] if ci else np.nan,
        "high_tier_defined": hm is not None,
        "note": "no 2+ rows -- High tier undefined, >=1 task only" if hm is None else "",
    })

h_sweep_summary = pd.DataFrame(h_summary_rows)
h_sweep_summary.to_csv(TABLES_DIR / "82_h_sweep_robustness_summary.csv", index=False)
h_sweep_summary

,arm,n_rows,2plus_rate,ge1_rate,macro_F1,macro_F1_majority_baseline,balanced_acc,balanced_acc_majority_baseline,high_precision,high_recall,high_lift,ci_precision_minus_prevalence,ci_low,ci_high,high_tier_defined,note
0,h=3 (n=8),2700,0.000000,0.221481,0.665899,0.428370,0.699061,0.500000,NaN,NaN,NaN,NaN,NaN,NaN,False,"no 2+ rows -- High tier undefined, >=1 task only"
1,h=5 (n=8),2700,0.002593,0.328889,0.408886,0.258812,0.406138,0.333333,0.0000,0.0,0.0,-0.001262,-0.003704,0.000000,True,
2,h=7 (n=8),2400,0.009167,0.418750,0.385270,0.234623,0.425297,0.333333,0.0125,0.2,1.8,0.004943,-0.011111,0.029203,True,


In [13]:
# h=3's >=1 task, reported explicitly since High is undefined there
h3_ge1_precision, h3_ge1_prevalence, h3_ge1_lift = None, None, None
test3 = h_arms[3]["test"]
pred3 = h_arms[3]["pred"]
ge1_actual = test3.target >= 1
ge1_pred = pred3 >= 1
h3_ge1_precision = precision_score(ge1_actual, ge1_pred, zero_division=0)
h3_ge1_recall = recall_score(ge1_actual, ge1_pred, zero_division=0)
print(f"h=3 (n=8): no 2+ rows exist in this label recipe (future_stagnant_count in {{0,1}} only) -- "
      "High tier is structurally undefined, not just underpowered.")
print(f"  >=1 task only: precision={h3_ge1_precision:.4f}, recall={h3_ge1_recall:.4f}, "
      f"prevalence={ge1_actual.mean():.4f}")

h=3 (n=8): no 2+ rows exist in this label recipe (future_stagnant_count in {0,1} only) -- High tier is structurally undefined, not just underpowered.
  >=1 task only: precision=0.4518, recall=0.6700, prevalence=0.2506


## J. Cross-arm feature stability -- top-10 |coefficient| across the n-sweep

Comparing the top-10 features by absolute logistic-regression coefficient (summed across the
one-vs-rest classes) across the n-sweep arms. Coefficients are on the standardized feature scale
(post `StandardScaler`), so magnitudes are comparable within an arm and roughly comparable across
arms since the same features and scaler recipe are used everywhere.

In [14]:
def top_features(arm, k=10):
    clf = arm["pipe"].named_steps["clf"]
    coef = np.abs(clf.coef_).sum(axis=0)  # sum abs coef across the 3 one-vs-rest classes
    s = pd.Series(coef, index=FEATURES_A).sort_values(ascending=False)
    return s.head(k)

top_feat_by_arm = {f"n={n}": top_features(arm) for n, arm in n_arms.items()}
feature_stability = pd.DataFrame({arm: s.index for arm, s in top_feat_by_arm.items()})
feature_stability.index = [f"rank_{i+1}" for i in range(len(feature_stability))]
feature_stability.to_csv(TABLES_DIR / "83_feature_stability_top10_by_arm.csv")
feature_stability

,n=2,n=3,n=4,n=6,n=8
rank_1,mastery_progression_rate_delta1,is_first_snapshot,is_first_snapshot,engagement_decline_rate,frustration_slope
rank_2,mastery_progression_rate_lag1,score_avg_recent_delta1,score_trend_slope_lag1,engagement_x_frustration,stagnant_skill_count_rollmean_prior
rank_3,mastery_progression_rate,score_trend_slope_delta1,error_recurrence_rate_rollmean_prior,score_trend_slope_rollmean_prior,zero_accuracy_window
rank_4,number_sense_mastery_lag1,score_avg_recent_lag1,engagement_decline_rate,error_recurrence_rate_rollmean_prior,score_trend_slope
rank_5,number_sense_mastery,mastery_progression_rate_delta1,sessions_observed,hint_intensity,score_trend_slope_lag1
rank_6,sessions_observed,score_trend_slope_lag1,score_trend_slope_rollmean_prior,score_trend_slope_lag1,frustration_mean_rollmean_prior
rank_7,number_sense_mastery_rollmean_prior,sessions_observed,score_trend_slope_delta1,error_recurrence_rate_lag1,error_recurrence_rate_rollmean_prior
rank_8,score_avg_recent,score_trend_slope,score_trend_slope,is_first_snapshot,hint_intensity
rank_9,weakest_skill_persistence,number_sense_mastery,number_sense_mastery,rt_accuracy_tradeoff,difficulty_floor_hits_missing
rank_10,hint_intensity_x_accuracy,slope_censored_count,stagnant_skill_count_rollmean_prior,number_sense_mastery_rollmean_prior,engagement_decline_rate


In [15]:
# how many of the n=4 (primary) top-10 features also appear in each other arm's top-10
primary_top10 = set(top_feat_by_arm["n=4"].index)
overlap_rows = []
for arm_name, s in top_feat_by_arm.items():
    overlap = len(primary_top10 & set(s.index))
    overlap_rows.append({"arm": arm_name, "overlap_with_n4_top10": overlap})
overlap_table = pd.DataFrame(overlap_rows)
overlap_table

,arm,overlap_with_n4_top10
0,n=2,2
1,n=3,6
2,n=4,10
3,n=6,5
4,n=8,5


## K. Summary table -- arm x (macro-F1, High lift, High CI, per-population lift, verdict)

In [16]:
def verdict_for_arm(n, arm):
    if n == 2:
        return "EXCLUDED (sensitivity-only, near-degenerate label, DATASET_CARD Sec6)"
    hm, ci = arm["high_metrics"], arm["bootstrap_ci"]
    if hm is None or ci is None:
        return "High tier undefined"
    ci_excludes_zero = ci["ci_low"] > 0
    pop = arm["per_pop"]
    dd_row = pop[pop.population == "DD"]
    dd_lift = dd_row["lift"].iloc[0] if len(dd_row) else np.nan
    dd_ok = (dd_lift == dd_lift) and dd_lift >= 1.0
    if ci_excludes_zero and dd_ok:
        return "CI excludes zero AND DD lift >= 1 -- candidate for further review"
    reasons = []
    if not ci_excludes_zero:
        reasons.append("CI includes zero")
    if not dd_ok:
        reasons.append(f"DD lift {dd_lift:.2f} < 1 (inverts for DD)" if dd_lift == dd_lift else "DD lift undefined")
    return "NOT shippable -- " + "; ".join(reasons)

final_summary_rows = []
for n, arm in n_arms.items():
    hm, ci = arm["high_metrics"], arm["bootstrap_ci"]
    pop = arm["per_pop"]
    def pop_lift(p):
        row = pop[pop.population == p]
        return row["lift"].iloc[0] if len(row) else np.nan
    final_summary_rows.append({
        "arm": f"n={n} (h=5)", "macro_F1": f1_score(arm["test"].target, arm["pred"], average="macro"),
        "high_lift": hm["lift"] if hm else np.nan,
        "ci_low": ci["ci_low"] if ci else np.nan, "ci_high": ci["ci_high"] if ci else np.nan,
        "dd_lift": pop_lift("DD"), "mld_lift": pop_lift("MLD"), "td_lift": pop_lift("TD"),
        "verdict": verdict_for_arm(n, arm),
    })
for h, arm in h_arms.items():
    hm, ci = arm["high_metrics"], arm["bootstrap_ci"]
    pop = arm["per_pop"]
    def pop_lift(p):
        row = pop[pop.population == p]
        return row["lift"].iloc[0] if len(row) else np.nan
    final_summary_rows.append({
        "arm": f"h={h} (n=8, robustness only)", "macro_F1": f1_score(arm["test"].target, arm["pred"], average="macro"),
        "high_lift": hm["lift"] if hm else np.nan,
        "ci_low": ci["ci_low"] if ci else np.nan, "ci_high": ci["ci_high"] if ci else np.nan,
        "dd_lift": pop_lift("DD"), "mld_lift": pop_lift("MLD"), "td_lift": pop_lift("TD"),
        "verdict": "robustness check only, not a product arm" if hm is not None else "High tier undefined (no 2+ rows)",
    })

final_summary = pd.DataFrame(final_summary_rows)
final_summary.to_csv(TABLES_DIR / "84_arm_recommendation_summary.csv", index=False)
final_summary

,arm,macro_F1,high_lift,ci_low,ci_high,dd_lift,mld_lift,td_lift,verdict
0,n=2 (h=5),0.446584,1.345196,0.008956,0.237678,0.989011,2.305419,1.238556,"EXCLUDED (sensitivity-only, near-degenerate la..."
1,n=3 (h=5),0.426749,1.274336,-0.030601,0.108642,0.803571,2.600000,1.209872,NOT shippable -- CI includes zero; DD lift 0.8...
2,n=4 (h=5),0.423929,1.588235,-0.022780,0.106691,0.900000,5.571429,0.922959,NOT shippable -- CI includes zero; DD lift 0.9...
3,n=6 (h=5),0.375520,0.692308,-0.023457,0.021088,0.000000,0.000000,0.793421,NOT shippable -- CI includes zero; DD lift 0.0...
4,n=8 (h=5),0.408886,0.000000,-0.003704,0.000000,NaN,NaN,0.000000,NOT shippable -- CI includes zero; DD lift und...
5,"h=3 (n=8, robustness only)",0.665899,NaN,NaN,NaN,NaN,NaN,NaN,High tier undefined (no 2+ rows)
6,"h=5 (n=8, robustness only)",0.408886,0.000000,-0.003704,0.000000,NaN,NaN,0.000000,"robustness check only, not a product arm"
7,"h=7 (n=8, robustness only)",0.385270,1.800000,-0.011111,0.029203,0.000000,NaN,2.061538,"robustness check only, not a product arm"


## L. Does ANY arm support a shippable High tier?

A CI that excludes zero on (precision − prevalence) AND a per-population lift ≥1 for every
population (no inversion, especially not for DD) would be the bar for "credible evidence of a
shippable High tier." The table above (`84_arm_recommendation_summary.csv`) is checked
programmatically below.

In [17]:
shippable_arms = final_summary[final_summary.verdict.str.contains("candidate", case=False, na=False)]
if len(shippable_arms):
    print("Arm(s) meeting the credible-evidence bar (CI excludes zero AND no population inversion):")
    print(shippable_arms[["arm", "high_lift", "ci_low", "ci_high", "dd_lift"]].to_string(index=False))
else:
    print("NO arm in this sweep meets the credible-evidence bar for a shippable High tier.")
    print("This is a legitimate conclusion, not a gap in the analysis -- see the Findings below.")

NO arm in this sweep meets the credible-evidence bar for a shippable High tier.
This is a legitimate conclusion, not a gap in the analysis -- see the Findings below.


## Findings

### n=3 headline result

n=3 raises the 2+ base rate to ~13.9% (vs the primary n=4 arm's 6.0%), landing inside
DATASET_CARD.md Sec6's expected 8-15% "High" band. The macro-F1, High-tier precision/recall/lift and
the **learner-clustered bootstrap 95% CI on (High-tier precision − prevalence)** for n=3 are in
`81_n_sweep_headline_summary.csv` and printed in Sec H above -- read that CI directly: if it still
contains zero, a higher base rate alone did not buy credible evidence, matching the pattern
NB06/NB05b found repeatedly (a headline ratio that looks like signal and dissolves under a
learner-clustered interval). The per-population (DD/MLD/TD) lift for n=3 is in Sec F / the printed
table above and in `83`-adjacent per-arm output -- check specifically whether DD lift recovers to
≥1 (NB06's blocking finding was DD lift 0.75, i.e. the High tier was *worse than flagging everyone*
for DD learners at n=4/h=5).

### h-sweep as a robustness statement

The h-sweep (h=3, h=5≡sweepn8, h=7, all at n=8) sits at the weakest-signal n in the whole study
(2+ prevalence ~0.000-0.009). It is reported as a **pipeline robustness check only** — does the same
pipeline degrade gracefully as horizon changes — and not as a candidate product arm; the near-zero
High-tier metrics at n=8 are not evidence either for or against shippability, they are evidence the
pipeline does not break at a different horizon. `h=3` has zero 2+ rows in this dataset at n=8, so
only the ≥1 task is reported for it (Sec I) — a High tier cannot be evaluated against a class that
does not exist in the data, not merely one that is rare. The matched-n=8 monotonicity check (Sec B)
passed with zero violations, confirming `future_stagnant_count` behaves sensibly (non-decreasing) as
`h` grows once `n` is held fixed — the comparison that is *not* n/h-confounded.

### Cross-arm feature stability

`83_feature_stability_top10_by_arm.csv` and the overlap-with-n=4 count in Sec J show whether the
same handful of behavioural features (score trend, mastery progression, engagement/frustration
signals) dominate regardless of which `n` threshold defines "stagnant," or whether the ranking
reorders substantially as the label's base rate shifts. A stable top-10 across arms would say the
*model's story* about what predicts stagnation does not depend on the contested `n` threshold, even
though the *label's usability* (base rate, High-tier lift) does.

### The ≥1 watch signal's robustness

`80_watch_signal_t3_lift_by_arm.csv` (Sec G) reports the lift of the "≥1 watch" signal at `t=3` for
every arm. This is the one finding NB06's findings doc concluded WAS supported for the primary arm
(lift 2.70 at t=3, decaying to ~1.0 by t=13). If this lift stays comfortably above 1.0 across the
n-sweep and h-sweep, that is evidence the early-detection watch signal — not the contested High
tier — is the robust part of this pipeline, consistent with `06_target_b_findings.md` Sec5's
recommendation to ship the watch list and drop the High tier.

### Recommendation — construct vs. metrics (owner decision 3-Q1)

Per Sec L: check whether ANY arm cleared the credible-evidence bar (bootstrap CI excludes zero AND
no population inverts, especially DD). **"None does" is a fully legitimate conclusion** if that is
what `84_arm_recommendation_summary.csv` shows — do not read a marginal metric improvement as a
manufactured win.

If n=3 *does* clear the bar, that is **not simply a "switch n=3 in" decision**. DATASET_CARD.md
Sec6 is explicit that `n=4` was chosen on a **construct** argument, not a metrics one: the trailing
window is `w=4`, so `n=4` is the minimum evidence density (one opportunity per session across the
window) at which "mastery is flat" is distinguishable from "the skill was barely practised." `n=3`
would mean flagging a skill flat on fewer opportunities than one per session — a construct-validity
question about what "stagnant" *means*, not a question this notebook's metrics can settle. **This is
flagged explicitly as owner decision 3-Q1, and is not decided here.** If the n=3 metrics are
materially better, the honest framing for the owner is: "n=3 may buy a usable High tier, at the cost
of a weaker construct argument for what counts as evidence of stagnation — that trade-off is yours
to make, not the model's."

n=2 is excluded from this decision entirely (sensitivity-only, near-degenerate ≥1 rate ~81.7%,
DATASET_CARD.md Sec6 warning) and should never be compared on raw metrics against any other arm.